In [ ]:
!pip install -r requirements.txt 

## Daft on Ray

### Init ray client and connect to the cluster

In [ ]:
import ray
ray.init("ray://8.152.158.25:10001", runtime_env={"pip": ["daft"]}, ignore_reinit_error=True)

### Run daft expresssions on ray cluster

In [ ]:
import daft

# Starts the Ray client and tells Daft to use Ray to execute queries
# If ray.init() has already been called, it uses the existing client
# daft.context.set_runner_ray()

df = daft.from_pydict({
    "a": [3, 2, 5, 6, 1, 4],
    "b": [True, False, False, True, True, False]
})
df = df.where(df["b"]).sort(df["a"])
df.collect()

### Run daft on ray to talk to lancedb

In [ ]:
import lancedb
import numpy as np


df = daft.from_pydict({
    "text": ["hello world", "openai builds daft", "vector db with LanceDB"],
    "id": [1, 2, 3]
})
df = df.with_column("word_count", df["text"].str.length())

pdf = df.to_pandas()
pdf["vector"] = pdf["text"].apply(lambda x: np.random.rand(128).tolist())
print(pdf.head())

print("Create a lancedb table...")
db = lancedb.connect("my_lance_db")
table = db.create_table("my_table", data=pdf, mode="overwrite")

# Run a similarity search
print("Running a similarity search...")
query_vec = np.random.rand(128).tolist()
results = table.search(query_vec).limit(2).to_pandas()
print(results)

In [ ]:
ray.shutdown()